# EfficientNet-B4 for Diabetic Retinopathy Detection
**Master's Capstone — Explainable AI for DR**

This notebook trains EfficientNet-B4 on the EyePACS dataset (88K fundus images).

**Before running:**
1. Set Runtime → Change runtime type → **T4 GPU**
2. Upload your `kaggle.json` API token when prompted in Cell 3
3. Run cells top-to-bottom

In [ ]:
# ── Cell 1: Check GPU ─────────────────────────────────────────────────────────
import torch

if not torch.cuda.is_available():
    raise RuntimeError('No GPU found! Go to Runtime → Change runtime type → T4 GPU')

gpu = torch.cuda.get_device_name(0)
mem = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU   : {gpu}')
print(f'VRAM  : {mem:.1f} GB')
print(f'PyTorch: {torch.__version__}')

In [ ]:
# ── Cell 2: Install dependencies ──────────────────────────────────────────────
# timm and kaggle are the only things Colab doesn't have by default
!pip install timm kaggle -q
print('Dependencies ready.')

In [ ]:
# ── Cell 3: Mount Google Drive ────────────────────────────────────────────────
from google.colab import drive
import os

drive.mount('/content/drive')

# All outputs (checkpoints, curves, history) go here
DRIVE_DIR   = '/content/drive/MyDrive/dr_xai'
CKPT_DIR    = f'{DRIVE_DIR}/checkpoints'
METRICS_DIR = f'{DRIVE_DIR}/results/metrics'
FIGURES_DIR = f'{DRIVE_DIR}/results/figures'

for d in [CKPT_DIR, METRICS_DIR, FIGURES_DIR]:
    os.makedirs(d, exist_ok=True)

print(f'Drive mounted. Outputs → {DRIVE_DIR}')

In [ ]:
# ── Cell 4: Kaggle setup & dataset download ───────────────────────────────────
#
# STEP 1: Go to kaggle.com → Account → Create New Token → download kaggle.json
# STEP 2: Run this cell and upload the file when prompted
#
# Dataset used: diabetic-retinopathy-detection (EyePACS, 2015 Kaggle competition)
#   train/ : ~35K labeled .jpeg images
#   trainLabels.csv : columns  image | level
#
# If you have the 88K version, update KAGGLE_DATASET below accordingly.

from google.colab import files

print('Upload your kaggle.json ...')
uploaded = files.upload()   # select kaggle.json from your computer

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
print('kaggle.json installed.')

# ── Download ──────────────────────────────────────────────────────────────────
DATA_ROOT = '/content/eyepacs'
os.makedirs(DATA_ROOT, exist_ok=True)

# Change this if you have a different dataset slug
KAGGLE_COMPETITION = 'diabetic-retinopathy-detection'

print(f'Downloading {KAGGLE_COMPETITION} ...')
!kaggle competitions download -c {KAGGLE_COMPETITION} -p {DATA_ROOT} --quiet
print('Unzipping ...')
!unzip -q {DATA_ROOT}/{KAGGLE_COMPETITION}.zip -d {DATA_ROOT}
print('Done.')

# Verify
!echo 'train images:' && ls {DATA_ROOT}/train | wc -l
!echo 'CSV:' && head -3 {DATA_ROOT}/trainLabels.csv

In [ ]:
# ── Cell 5: Configuration ─────────────────────────────────────────────────────
# Edit these if your dataset has different column names or file layout

import os

# Paths
TRAIN_IMG_DIR = f'{DATA_ROOT}/train'           # folder with .jpeg images
TRAIN_CSV     = f'{DATA_ROOT}/trainLabels.csv' # image | level

# CSV column names  ← change for other datasets
IMAGE_COL = 'image'   # stem of image filename, no extension
LABEL_COL = 'level'   # DR grade 0-4

# Image extension (EyePACS uses .jpeg)
IMG_EXT = '.jpeg'

# Hyperparameters
NUM_CLASSES  = 5
TARGET_SIZE  = 512
VAL_SPLIT    = 0.15      # 15% validation
LR           = 1e-4
BATCH_SIZE   = 32        # GPU can handle 32 at 512×512
NUM_EPOCHS   = 20
NUM_WORKERS  = 4
SEED         = 42

CKPT_PATH    = f'{CKPT_DIR}/efficientnet_b4_best.pth'

print('Configuration:')
print(f'  Image dir  : {TRAIN_IMG_DIR}')
print(f'  CSV        : {TRAIN_CSV}')
print(f'  Classes    : {NUM_CLASSES}')
print(f'  Batch size : {BATCH_SIZE}')
print(f'  Epochs     : {NUM_EPOCHS}')
print(f'  Workers    : {NUM_WORKERS}')

In [ ]:
# ── Cell 6: Preprocessing (from src/preprocessing.py) ────────────────────────

import cv2
import numpy as np
from PIL import Image


def load_image(image_path):
    img = Image.open(image_path).convert('RGB')
    return np.array(img)


def resize_image(image, target_size=512):
    return cv2.resize(image, (target_size, target_size), interpolation=cv2.INTER_AREA)


def normalize_image(image):
    return image.astype(np.float32) / 255.0


def preprocess_image(image_path, target_size=512):
    img = load_image(image_path)
    img = resize_image(img, target_size)
    img = normalize_image(img)
    return img


print('Preprocessing functions loaded.')

# Quick smoke-test on one image
import glob
sample = glob.glob(f'{TRAIN_IMG_DIR}/*{IMG_EXT}')[:1]
if sample:
    arr = preprocess_image(sample[0], TARGET_SIZE)
    print(f'Sample shape : {arr.shape}  range: [{arr.min():.2f}, {arr.max():.2f}]')

In [ ]:
# ── Cell 7: Dataset class (from src/dataset.py, adapted for EyePACS) ──────────

import torch
from torch.utils.data import Dataset, DataLoader, random_split, Subset
import pandas as pd
from pathlib import Path
from torchvision import transforms


class DRDataset(Dataset):
    """
    Diabetic Retinopathy Dataset.
    Works with any CSV that has an image-ID column and a label column.
    """

    def __init__(self, image_dir, csv_file,
                 image_col='image', label_col='level',
                 img_ext='.jpeg',
                 transform=None, target_size=512):
        self.image_dir   = Path(image_dir)
        self.transform   = transform
        self.target_size = target_size
        self.img_ext     = img_ext

        df = pd.read_csv(csv_file)
        df.columns = df.columns.str.strip()
        self.labels_df = df[[image_col, label_col]].copy()
        self.labels_df.columns = ['image_id', 'label']
        self.labels_df = self.labels_df.set_index('image_id')

        # Only keep rows whose image file actually exists
        self.image_ids = [
            stem for stem in self.labels_df.index
            if (self.image_dir / f'{stem}{img_ext}').exists()
        ]
        print(f'Dataset: {len(self.image_ids)} images found (CSV had {len(self.labels_df)} rows)')

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        stem  = self.image_ids[idx]
        path  = self.image_dir / f'{stem}{self.img_ext}'
        image = preprocess_image(str(path), self.target_size)
        image = torch.from_numpy(image).permute(2, 0, 1).float()  # (C,H,W)

        if self.transform:
            image = self.transform(image)

        label = int(self.labels_df.loc[stem, 'label'])
        return image, torch.tensor(label, dtype=torch.long)


# Augmentation for training
TRAIN_TRANSFORM = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
])

print('Dataset class loaded.')

In [ ]:
# ── Cell 8: Build DataLoaders ─────────────────────────────────────────────────

import numpy as np
from sklearn.utils.class_weight import compute_class_weight

torch.manual_seed(SEED)
np.random.seed(SEED)

full_ds = DRDataset(
    image_dir=TRAIN_IMG_DIR,
    csv_file=TRAIN_CSV,
    image_col=IMAGE_COL,
    label_col=LABEL_COL,
    img_ext=IMG_EXT,
    target_size=TARGET_SIZE,
)

n       = len(full_ds)
n_val   = max(1, int(n * VAL_SPLIT))
n_train = n - n_val

generator = torch.Generator().manual_seed(SEED)
train_ds, val_ds = random_split(full_ds, [n_train, n_val], generator=generator)

# Attach augmentation only to the training split
train_ds.dataset.transform = TRAIN_TRANSFORM

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print(f'Train: {n_train:,}  |  Val: {n_val:,}')

# Class weights (computed from all labels — fast, no image loading)
all_labels = np.array([int(full_ds.labels_df.loc[stem, 'label'])
                        for stem in full_ds.image_ids])

print('\nLabel distribution:')
for cls in range(NUM_CLASSES):
    count = (all_labels == cls).sum()
    print(f'  Grade {cls}: {count:,} ({100*count/len(all_labels):.1f}%)')

present = np.unique(all_labels)
partial = compute_class_weight('balanced', classes=present, y=all_labels)
class_weights_np = np.ones(NUM_CLASSES, dtype=np.float32)
for cls, w in zip(present, partial):
    class_weights_np[cls] = w

device        = torch.device('cuda')
class_weights = torch.tensor(class_weights_np, dtype=torch.float32, device=device)
print(f'\nClass weights: {np.round(class_weights_np, 3)}')

In [ ]:
# ── Cell 9: EfficientNet-B4 model ─────────────────────────────────────────────

import torch.nn as nn
import timm


class DRClassifier(nn.Module):
    def __init__(self, num_classes=5, pretrained=True, dropout=0.3):
        super().__init__()
        self.backbone = timm.create_model(
            'efficientnet_b4',
            pretrained=pretrained,
            num_classes=0,   # remove default head
        )
        self.classifier = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(self.backbone.num_features, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.backbone(x))


model     = DRClassifier(num_classes=NUM_CLASSES, pretrained=True).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

# Cosine LR scheduler (optional but helpful for 20 epochs)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params    : {total:,}')
print(f'Trainable params: {trainable:,}')

In [ ]:
# ── Cell 10: Training & validation loops ──────────────────────────────────────

from tqdm.notebook import tqdm
from sklearn.metrics import roc_auc_score


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = correct = total = 0
    for images, labels in tqdm(loader, desc='  Train', leave=False):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(labels)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += len(labels)
    return total_loss / total, correct / total


@torch.no_grad()
def validate(model, loader, criterion, device, num_classes):
    model.eval()
    total_loss = correct = total = 0
    all_probs, all_labels = [], []
    for images, labels in tqdm(loader, desc='  Val  ', leave=False):
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        loss   = criterion(logits, labels)
        all_probs.append(torch.softmax(logits, dim=1).cpu().numpy())
        all_labels.append(labels.cpu().numpy())
        total_loss += loss.item() * len(labels)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += len(labels)
    all_probs  = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)
    try:
        auroc = roc_auc_score(all_labels, all_probs,
                              multi_class='ovr', average='macro',
                              labels=list(range(num_classes)))
    except ValueError:
        auroc = float('nan')
    return total_loss / total, correct / total, auroc


print('Training functions ready.')

In [ ]:
# ── Cell 11: Full training run (20 epochs) ────────────────────────────────────

import pandas as pd

history    = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'val_auroc': []}
best_auroc = -1.0

print(f'Training EfficientNet-B4 for {NUM_EPOCHS} epochs on {n_train:,} images')
print(f'Checkpoints → {CKPT_PATH}\n')

for epoch in range(1, NUM_EPOCHS + 1):
    print(f'Epoch {epoch}/{NUM_EPOCHS}')

    train_loss, train_acc             = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss,   val_acc,   val_auroc  = validate(model, val_loader, criterion, device, NUM_CLASSES)
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_auroc'].append(val_auroc if not np.isnan(val_auroc) else 0.0)

    auroc_str = f'{val_auroc:.4f}' if not np.isnan(val_auroc) else 'N/A'
    print(f'  Train  loss={train_loss:.4f}  acc={train_acc:.3f}')
    print(f'  Val    loss={val_loss:.4f}  acc={val_acc:.3f}  AUROC={auroc_str}')

    if not np.isnan(val_auroc) and val_auroc > best_auroc:
        best_auroc = val_auroc
        torch.save({
            'epoch': epoch,
            'model_state_dict':     model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_auroc': val_auroc,
            'val_acc':   val_acc,
        }, CKPT_PATH)
        print(f'  ✓ Best checkpoint saved  (AUROC={best_auroc:.4f})')

    print()

# Save history
history_path = f'{METRICS_DIR}/training_history.csv'
pd.DataFrame(history).to_csv(history_path, index=False)
print(f'History saved → {history_path}')
print(f'Best Val AUROC: {best_auroc:.4f}')

In [ ]:
# ── Cell 12: Plot training curves ─────────────────────────────────────────────

import matplotlib.pyplot as plt

epochs = range(1, len(history['train_loss']) + 1)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('EfficientNet-B4 Training — EyePACS', fontsize=13)

axes[0].plot(epochs, history['train_loss'], label='Train')
axes[0].plot(epochs, history['val_loss'],   label='Val')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()

axes[1].plot(epochs, history['train_acc'], label='Train')
axes[1].plot(epochs, history['val_acc'],   label='Val')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].legend()

axes[2].plot(epochs, history['val_auroc'], color='green', marker='o', markersize=4)
axes[2].axhline(y=0.75, color='red', linestyle='--', label='Target 0.75')
axes[2].axhline(y=0.80, color='orange', linestyle='--', label='Target 0.80')
axes[2].set_title('Val AUROC'); axes[2].set_xlabel('Epoch'); axes[2].legend()

plt.tight_layout()
curve_path = f'{FIGURES_DIR}/training_curves.png'
plt.savefig(curve_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Curves saved → {curve_path}')

In [ ]:
# ── Cell 13: Evaluation on held-out val set ───────────────────────────────────
#
# Loads the best checkpoint and runs a full evaluation.

from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, ConfusionMatrixDisplay
)

# Load best model
ckpt = torch.load(CKPT_PATH, map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
print(f"Loaded checkpoint from epoch {ckpt['epoch']}  (AUROC={ckpt['val_auroc']:.4f})")

# Collect predictions
model.eval()
all_probs, all_preds, all_labels = [], [], []

with torch.no_grad():
    for images, labels in tqdm(val_loader, desc='Evaluating'):
        images = images.to(device)
        logits = model(images)
        probs  = torch.softmax(logits, dim=1).cpu().numpy()
        preds  = logits.argmax(1).cpu().numpy()
        all_probs.append(probs)
        all_preds.append(preds)
        all_labels.append(labels.numpy())

all_probs  = np.concatenate(all_probs)
all_preds  = np.concatenate(all_preds)
all_labels = np.concatenate(all_labels)

# Metrics
auroc = roc_auc_score(all_labels, all_probs, multi_class='ovr',
                      average='macro', labels=list(range(NUM_CLASSES)))

print(f'\n=== Evaluation Results ===')
print(f'Val AUROC (macro OvR): {auroc:.4f}')
print(f'Target range: 0.75 – 0.80')
print()

CLASS_NAMES = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative']
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))

# Confusion matrix
fig, ax = plt.subplots(figsize=(7, 6))
cm = confusion_matrix(all_labels, all_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
disp.plot(ax=ax, colorbar=False)
ax.set_title('Confusion Matrix — Validation Set')
plt.tight_layout()
cm_path = f'{FIGURES_DIR}/confusion_matrix.png'
plt.savefig(cm_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Confusion matrix saved → {cm_path}')

In [ ]:
# ── Cell 14: Save final summary to Drive ──────────────────────────────────────

summary = {
    'best_epoch':   ckpt['epoch'],
    'best_val_auroc': best_auroc,
    'final_val_acc':  float(all_preds == all_labels).mean(),  # type: ignore
    'num_classes':  NUM_CLASSES,
    'target_size':  TARGET_SIZE,
    'batch_size':   BATCH_SIZE,
    'lr':           LR,
    'epochs':       NUM_EPOCHS,
    'train_images': n_train,
    'val_images':   n_val,
}

import json
summary_path = f'{DRIVE_DIR}/training_summary.json'
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)

print('All outputs saved to Google Drive:')
print(f'  Checkpoint      : {CKPT_PATH}')
print(f'  Training history: {METRICS_DIR}/training_history.csv')
print(f'  Training curves : {FIGURES_DIR}/training_curves.png')
print(f'  Confusion matrix: {FIGURES_DIR}/confusion_matrix.png')
print(f'  Summary JSON    : {summary_path}')
print(f'\nBest Val AUROC: {best_auroc:.4f}')